# Sensitivity analysis: recommender scoring modes

In this notebook, we will explore ten "personas" and try out the three different `scoring_mode` values (`similarity`, `weighted_average`, `cosine`). We will explore the returned top 3 metros per persona per mode. Then each dimension will be adjusted slightly by plus or minus 0.5 (clipped to 0 through 5), one dimension at a time and we will save the top 3 metros each time. Through this analysis we will manually validate that the returned reccomended cities seem sound based on what we know about the cities, as well as test how sensitive our reccomendation / scoring model is to slight changes in the inputs.  I will use the tables and plots below to compare modes and how stable rankings are to small preference tweaks.

In [1]:
# Set working directory to project root and ensure src/ import path.
from pathlib import Path
import os
import sys

repo_root = Path.cwd()
if repo_root.name == "exploratory_notebooks":
    repo_root = repo_root.parent
os.chdir(repo_root)

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("CWD:", Path.cwd())

CWD: c:\Users\sosettle\FINAL CHECK\movesmart


In [3]:
# Notebook-only patch: stub semantic_search so recommender import doesn't load torch/sentence-transformers
import types
import sys
stub = types.ModuleType("src.semantic_search")
stub.search = lambda query, top_k=10: []  # returns no text matches
sys.modules["src.semantic_search"] = stub

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src.recommender import recommend_cities, DIMENSION_SCORE_COLS, SCORE_MAX

df = pd.read_csv(
    "data/final/Final_Enriched_Dataset.csv",
    dtype={"cbsa_code": str},
)

In [5]:
df.head()

,cbsa_code,cbsa_name,city,state,cbsa_type,contains_imputed,MedianHouseholdIncome_B19013,TotalPopulation_B01003,MedianAge_B01002,TotalPovertyUniverse_B17001,...,cluster_tradeoffs,sub_cluster,sub_cluster_text,sub_cluster_key_traits,sub_cluster_candidate_name_1,sub_cluster_candidate_name_2,sub_cluster_candidate_name_3,sub_cluster_final_name,sub_cluster_description,sub_cluster_tradeoffs
0,10100,"Aberdeen, SD Micro Area",Aberdeen,SD,Micro,0,70761,42112,38.2,40877,...,Tough winters; amenities can be more limited t...,20,Less diverse / Higher unemployment,annual_snowfall: higher vs avg (10.73 vs 9.316...,Cold-Season Worktowns,Seasonal Recreation & Work Metros,Snow-Season Blue-Collar Mix,Cold-Season Worktowns,Seasonal metros with colder winters and a work...,Winter + lifestyle-health risk signals.
1,10140,"Aberdeen, WA Micro Area",Aberdeen,WA,Micro,0,63539,76397,44.7,73378,...,High cost of living; less starter-city friendly.,18,Fast job growth / Car-dependent,job_growth: higher vs avg (0.05782 vs 0.02067)...,Growing Older-Skew Metros,Booming but Aging Cities,Expansion-Phase Heartland,Booming but Aging Cities,Growing metros that still skew older and show ...,Growth upside; health profile not as strong.
2,10180,"Abilene, TX Metro Area",Abilene,TX,Metro,0,66464,178244,34.4,166018,...,"Energy and youth skew, but less stability/amen...",21,High-rent / More educated,BlackAlone_B02001: higher vs avg (6.494e+04 vs...,Young Government & Nightlife Hubs,Active Risk-Tradeoff Cities,Walkable Young Metros,Active Risk-Tradeoff Cities,Younger metros with more activity and walkabil...,Lifestyle/access vs safety.
3,10220,"Ada, OK Micro Area",Ada,OK,Micro,0,62564,38158,37.5,36961,...,"Energy and youth skew, but less stability/amen...",1,Low-rent / Less diverse,annual_snowfall: lower vs avg (1.599 vs 9.316)...,Stagnating Worktowns,Slow-Growth Factory Markets,Post-Boom Work-First Metros,Stagnating Worktowns,Lower-growth metros with a work-first footprin...,"Often more affordable, but fewer fast-growth o..."
4,10300,"Adrian, MI Micro Area",Adrian,MI,Micro,0,67013,98823,42.2,93980,...,Tough winters; amenities can be more limited t...,20,Less diverse / Higher unemployment,annual_snowfall: higher vs avg (10.73 vs 9.316...,Cold-Season Worktowns,Seasonal Recreation & Work Metros,Snow-Season Blue-Collar Mix,Cold-Season Worktowns,Seasonal metros with colder winters and a work...,Winter + lifestyle-health risk signals.


#### Set up Personas

In [6]:
# set constants
SCORING_MODES = ("similarity", "weighted_average", "cosine")
TOP_N = 5
DELTA = .25

#  gemerate ten personas — values listed in DIMENSION_SCORE_COLS order
ORDER = DIMENSION_SCORE_COLS


def P(vals):
    return dict(zip(ORDER, vals))


def generate_personas(n_personas=1000, n_features=None, seed=42):
    np.random.seed(seed)
    dim_cols = list(DIMENSION_SCORE_COLS)
    n_features = n_features or len(dim_cols)
    if n_features != len(dim_cols):
        raise ValueError(
            f"n_features ({n_features}) must equal len(DIMENSION_SCORE_COLS) ({len(dim_cols)})"
        )
    personas = []
    for i in range(1, n_personas + 1):
        persona_type = np.random.choice(["balanced", "spiky", "mixed"])
        if persona_type == "balanced":
            prefs = np.clip(np.random.normal(loc=2.5, scale=0.7, size=n_features), 0, 5)
        elif persona_type == "spiky":
            prefs = np.ones(n_features) * np.random.uniform(1.5, 3)
            spike_indices = np.random.choice(
                n_features, size=np.random.randint(1, 3), replace=False
            )
            prefs[spike_indices] = np.random.uniform(4, 5, size=len(spike_indices))
        else:  # mixed
            prefs = np.random.uniform(0, 5, size=n_features)
        prefs = np.round(prefs, 2)
        personas.append(
            {
                "persona_id": i,
                "prefs": prefs.tolist(),
                "user_inputs": dict(zip(dim_cols, prefs.tolist())),
            }
        )
    return personas

PERSONAS = generate_personas(n_personas=100)
PERSONAS

[{'persona_id': 1,
  'prefs': [3.98, 0.92, 3.9, 2.98, 2.23, 0.5, 2.3, 1.67, 0.71, 3.25],
  'user_inputs': {'affordability_score': 3.98,
   'job_growth_score': 0.92,
   'safety_score': 3.9,
   'education_score': 2.98,
   'health_score': 2.23,
   'walkability_score': 0.5,
   'diversity_score': 2.3,
   'urban_score': 1.67,
   'weather_warmth_score': 0.71,
   'weather_mildness_score': 3.25}},
 {'persona_id': 2,
  'prefs': [2.09, 2.13, 2.1, 1.85, 0.67, 3.17, 3.07, 1.43, 2.2, 1.98],
  'user_inputs': {'affordability_score': 2.09,
   'job_growth_score': 2.13,
   'safety_score': 2.1,
   'education_score': 1.85,
   'health_score': 0.67,
   'walkability_score': 3.17,
   'diversity_score': 3.07,
   'urban_score': 1.43,
   'weather_warmth_score': 2.2,
   'weather_mildness_score': 1.98}},
 {'persona_id': 3,
  'prefs': [2.28, 3.93, 1.0, 2.57, 2.96, 0.23, 3.04, 0.85, 0.33, 4.74],
  'user_inputs': {'affordability_score': 2.28,
   'job_growth_score': 3.93,
   'safety_score': 1.0,
   'education_score': 2

##### Helper functions

In [7]:
# one-slider nudge
def perturb(prefs, dim, delta):
    if isinstance(prefs, dict):
        out = dict(prefs)
        out[dim] = float(np.clip(out[dim] + delta, 0.0, SCORE_MAX))
        return out

    out = list(prefs)
    idx = list(DIMENSION_SCORE_COLS).index(dim)
    out[idx] = float(np.clip(float(out[idx]) + delta, 0.0, SCORE_MAX))
    return out

# overlap of top-3 CBSA sets
def jaccard_topN(a, b):
    sa, sb = set(a), set(b)
    return len(sa & sb) / len(sa | sb)

In [8]:
PERSONAS[0]['prefs']

[3.98, 0.92, 3.9, 2.98, 2.23, 0.5, 2.3, 1.67, 0.71, 3.25]

##### Run baselines for each persona

In [11]:
# run baseline grid
rows = []

for p in PERSONAS:
    # recommend_cities expects a mapping {dimension_score_col: rating}
    user_inputs = dict(zip(DIMENSION_SCORE_COLS, p["prefs"]))

    for mode in SCORING_MODES:
        top = recommend_cities(
            df,
            user_query="",
            user_inputs=user_inputs,
            user_income=None,
            housing_mode="either",
            top_n=TOP_N,
            scoring_mode=mode,
        )

        for rank, (_, row) in enumerate(top.head(3).iterrows(), start=1):
            rows.append(
                {
                    "persona_id": p.get("persona_id"),
                    "persona_label": p.get("label", ""),
                    "scoring_mode": mode,
                    "rank": rank,
                    "cbsa_code": row.get("cbsa_code"),
                    "cbsa_name": row.get("cbsa_name"),
                    "city": row.get("city"),
                    "state": row.get("state"),
                    "recommendation_score": row.get("recommendation_score"),
                }
            )

baseline_top_n = pd.DataFrame(rows)
display(baseline_top_n.head(12))


,persona_id,persona_label,scoring_mode,rank,cbsa_code,cbsa_name,city,state,recommendation_score
0,1,,similarity,1,31380,"Macomb, IL Micro Area",Macomb,IL,4.86
1,1,,similarity,2,42900,"Seneca Falls, NY Micro Area",Seneca Falls,NY,4.86
2,1,,similarity,3,21060,"Elizabethtown, KY Metro Area",Elizabethtown,KY,4.86
3,1,,weighted_average,1,12060,"Atlanta-Sandy Springs-Roswell, GA Metro Area",Atlanta-Sandy Springs-Roswell,GA,3.71
4,1,,weighted_average,2,12020,"Athens-Clarke County, GA Metro Area",Athens-Clarke County,GA,3.69
5,1,,weighted_average,3,44260,"Starkville, MS Micro Area",Starkville,MS,3.67
6,1,,cosine,1,42900,"Seneca Falls, NY Micro Area",Seneca Falls,NY,4.79
7,1,,cosine,2,31380,"Macomb, IL Micro Area",Macomb,IL,4.77
8,1,,cosine,3,34380,"Mount Pleasant, MI Micro Area",Mount Pleasant,MI,4.73
9,2,,similarity,1,20100,"Dover, DE Metro Area",Dover,DE,4.86


##### Run Perturbs

In [13]:
rows = []

for p in PERSONAS:
    base = p["prefs"]  # list of 10 values, aligned to DIMENSION_SCORE_COLS

    for dim in DIMENSION_SCORE_COLS:
        for delta in (-DELTA, DELTA):
            prefs_p = perturb(base, dim, delta)  # still a list

            user_inputs_p = dict(zip(DIMENSION_SCORE_COLS, prefs_p))

            for mode in SCORING_MODES:
                top = recommend_cities(
                    df,
                    user_query="",
                    user_inputs=user_inputs_p,
                    user_income=None,
                    housing_mode="either",
                    top_n=TOP_N,
                    scoring_mode=mode,
                )

                for rank, (_, row) in enumerate(top.head(3).iterrows(), start=1):
                    rows.append(
                        {
                            "persona_id": p.get("persona_id"),
                            "perturbed_dimension": dim,
                            "delta": delta,
                            "scoring_mode": mode,
                            "rank": rank,
                            "cbsa_code": row.get("cbsa_code"),
                            "cbsa_name": row.get("cbsa_name"),
                            "city": row.get("city"),
                            "state": row.get("state"),
                            "recommendation_score": row.get("recommendation_score"),
                        }
                    )

perturbation_top_n = pd.DataFrame(rows)
display(perturbation_top_n.head(12))

,persona_id,perturbed_dimension,delta,scoring_mode,rank,cbsa_code,cbsa_name,city,state,recommendation_score
0,1,affordability_score,-0.25,similarity,1,25770,"Hemlock Farms, PA Micro Area",Hemlock Farms,PA,4.88
1,1,affordability_score,-0.25,similarity,2,21060,"Elizabethtown, KY Metro Area",Elizabethtown,KY,4.87
2,1,affordability_score,-0.25,similarity,3,42900,"Seneca Falls, NY Micro Area",Seneca Falls,NY,4.87
3,1,affordability_score,-0.25,weighted_average,1,12060,"Atlanta-Sandy Springs-Roswell, GA Metro Area",Atlanta-Sandy Springs-Roswell,GA,3.76
4,1,affordability_score,-0.25,weighted_average,2,12020,"Athens-Clarke County, GA Metro Area",Athens-Clarke County,GA,3.74
5,1,affordability_score,-0.25,weighted_average,3,45220,"Tallahassee, FL Metro Area",Tallahassee,FL,3.72
6,1,affordability_score,-0.25,cosine,1,42900,"Seneca Falls, NY Micro Area",Seneca Falls,NY,4.78
7,1,affordability_score,-0.25,cosine,2,31380,"Macomb, IL Micro Area",Macomb,IL,4.76
8,1,affordability_score,-0.25,cosine,3,25770,"Hemlock Farms, PA Micro Area",Hemlock Farms,PA,4.75
9,1,affordability_score,0.25,similarity,1,31380,"Macomb, IL Micro Area",Macomb,IL,4.86


In [14]:
# Add a column to distinguish scenario
baseline = baseline_top_n[['persona_id', 'scoring_mode', 'rank', 'cbsa_name', 'recommendation_score']].copy()
baseline["scenario"] = "baseline"

perturbation = perturbation_top_n[['persona_id', 'scoring_mode', 'rank', 'cbsa_name', 'recommendation_score']].copy()
perturbation["scenario"] = "perturbation"

# Combine
combined_df = pd.concat([baseline, perturbation], ignore_index=True)
combined_df.columns

Index(['persona_id', 'scoring_mode', 'rank', 'cbsa_name',
       'recommendation_score', 'scenario'],
      dtype='str')

In [15]:
combined_df

,persona_id,scoring_mode,rank,cbsa_name,recommendation_score,scenario
0,1,similarity,1,"Macomb, IL Micro Area",4.86,baseline
1,1,similarity,2,"Seneca Falls, NY Micro Area",4.86,baseline
2,1,similarity,3,"Elizabethtown, KY Metro Area",4.86,baseline
3,1,weighted_average,1,"Atlanta-Sandy Springs-Roswell, GA Metro Area",3.71,baseline
4,1,weighted_average,2,"Athens-Clarke County, GA Metro Area",3.69,baseline
...,...,...,...,...,...,...
18895,100,weighted_average,2,"San Jose-Sunnyvale-Santa Clara, CA Metro Area",4.18,perturbation
18896,100,weighted_average,3,"Austin-Round Rock-San Marcos, TX Metro Area",4.17,perturbation
18897,100,cosine,1,"Grand Junction, CO Metro Area",4.85,perturbation
18898,100,cosine,2,"Elmira, NY Metro Area",4.78,perturbation


In [16]:
def plot_recommendation_frequency(combined_df, persona_id):
    # Filter to selected persona
    df = combined_df[combined_df["persona_id"] == persona_id].copy()
    
    if df.empty:
        print(f"No data found for persona_id: {persona_id}")
        return
    
    # Count how many times each CBSA appears per scoring mode
    freq_df = (
        df.groupby(["scoring_mode", "cbsa_name"])
        .size()
        .reset_index(name="count")
    )
    
    # Optional: sort within each scoring mode
    freq_df = freq_df.sort_values(["scoring_mode", "count"], ascending=[True, False])
    
    # Plot
    fig = px.bar(
        freq_df,
        x="count",
        y="cbsa_name",
        color="scoring_mode",
        facet_col="scoring_mode",
        orientation="h",
        title=f"CBSA Recommendation Frequency for Persona {persona_id}",
        labels={
            "count": "Count",
            "cbsa_name": "CBSA",
            "scoring_mode": "SM"
        }
    )
    
    fig.update_layout(
        height=600,
        font=dict(size=10),  # overall font smaller
    )
    
    
    # Make each facet sorted nicely
    fig.for_each_yaxis(lambda yaxis: yaxis.update(categoryorder="total ascending"))
    
    fig.show()

In [17]:
for i in range(1,11):
    plot_recommendation_frequency(combined_df, persona_id=i)

NameError: name 'px' is not defined

##### Testing for Stability 

In [ ]:
from scipy.stats import spearmanr

def compute_spearman(df):
    results = []

    for (persona, method), group in df.groupby(["persona_id", "scoring_mode"]):
        baseline = group[group["scenario"] == "baseline"]
        perturb = group[group["scenario"] == "perturbation"]

        merged = baseline.merge(
            perturb,
            on="cbsa_name",
            suffixes=("_base", "_pert")
        )

        if len(merged) > 1:
            corr, _ = spearmanr(
                merged["rank_base"],
                merged["rank_pert"]
            )
            results.append({
                "persona_id": persona,
                "scoring_mode": method,
                "spearman_corr": corr
            })

    return pd.DataFrame(results)

In [ ]:
compute_spearman(combined_df).groupby('scoring_mode').mean()

,persona_id,spearman_corr
scoring_mode,,
cosine,50.5,0.798178
similarity,50.5,0.821968
weighted_average,50.5,0.905856


In [ ]:
compute_spearman(combined_df).groupby('scoring_mode').median()

,persona_id,spearman_corr
scoring_mode,,
cosine,50.5,0.846567
similarity,50.5,0.875704
weighted_average,50.5,0.941079


##### Testing for Consistency

In [ ]:
def jaccard_similarity(a, b):
    return len(a & b) / len(a | b)

def compute_topk_overlap(df, k=5):
    results = []

    for (persona, method), group in df.groupby(["persona_id", "scoring_mode"]):
        base = set(group[(group["scenario"]=="baseline") & (group["rank"]<=k)]["cbsa_name"])
        pert = set(group[(group["scenario"]=="perturbation") & (group["rank"]<=k)]["cbsa_name"])

        if base and pert:
            score = jaccard_similarity(base, pert)
            results.append({
                "persona_id": persona,
                "scoring_mode": method,
                "topk_overlap": score
            })

    return pd.DataFrame(results)

In [ ]:
compute_topk_overlap(combined_df).groupby('scoring_mode').mean()

,persona_id,topk_overlap
scoring_mode,,
cosine,50.5,0.639964
similarity,50.5,0.675893
weighted_average,50.5,0.790071


In [ ]:
compute_topk_overlap(combined_df).groupby('scoring_mode').median()

,persona_id,topk_overlap
scoring_mode,,
cosine,50.5,0.60
similarity,50.5,0.60
weighted_average,50.5,0.75


Between the cosine and similarity-based methods, the similarity approach demonstrated consistently higher rank stability (Spearman correlation) and greater top-k recommendation overlap across simulated personas. This suggests that the similarity method is more robust to small perturbations in user preferences, making it a more reliable choice for consistent recommendation outputs.

### T Test for Significance

In [ ]:
spearman_df = compute_spearman(combined_df)
overlap_df = compute_topk_overlap(combined_df)

# Pivot so rows = persona, columns = method
pivot_df = spearman_df.pivot(
    index="persona_id",
    columns="scoring_mode",
    values="spearman_corr"
)

cosine_scores = pivot_df["cosine"].values
similarity_scores = pivot_df["similarity"].values

In [ ]:
from scipy.stats import ttest_rel

t_stat, p_val = ttest_rel(cosine_scores, similarity_scores)

print("t-stat:", t_stat)
print("p-value:", p_val)

print("Cosine mean:", cosine_scores.mean())
print("Similarity mean:", similarity_scores.mean())

t-stat: -0.9387086268232393
p-value: 0.3501649530651527
Cosine mean: 0.7981778028150802
Similarity mean: 0.8219676785066037


While the similarity-based method shows slightly higher average rank stability than cosine similarity, the difference is not statistically significant (p = 0.35), suggesting that both methods perform comparably under perturbation.